# 03 — Encapsulation

Encapsulation is one of the four pillars of Object-Oriented Programming (the others being Abstraction, Inheritance, and Polymorphism).

**Encapsulation** means bundling data (attributes) and the methods that operate on that data into a single unit (a class), and **restricting direct access** to some of an object's components to protect the integrity of the data.

In this notebook we will cover:

1. Why encapsulation matters
2. Public, protected, and private attributes in Python
3. Name mangling
4. Getters and setters (the traditional way)
5. The `@property` decorator (the Pythonic way)
6. Validation inside setters
7. Read-only properties
8. A complete real-world example: `BankAccount`
9. Practice exercises


## 1. Why Encapsulation Matters

Without encapsulation, any part of a program can reach into an object and change its internal state directly, which can leave the object in an invalid or inconsistent state.

Encapsulation lets a class:

- **Hide internal implementation details** from the outside world
- **Control how attributes are read or modified** (validation, logging, computed values)
- **Change internal implementation later** without breaking code that uses the class

Let's see a problem first, then fix it with encapsulation.

In [ ]:
class BadAccount:
    def __init__(self, balance):
        self.balance = balance  # fully open attribute

acc = BadAccount(100)
acc.balance = -5000  # nothing stops this!
print(acc.balance)   # invalid state, but no error was raised


Nothing prevented us from setting a negative balance. That's the problem encapsulation solves.

## 2. Public, Protected, and Private Attributes

Python does not have strict access modifiers like `private`, `protected`, and `public` in Java or C++. Instead, it relies on **naming conventions**:

| Convention      | Syntax        | Meaning                                              |
|------------------|---------------|-------------------------------------------------------|
| Public           | `name`        | Accessible from anywhere                              |
| Protected        | `_name`       | *Convention only* — "internal use", please don't touch from outside |
| Private          | `__name`      | Name-mangled — harder (not impossible) to access from outside |

This is often summarized as: **"We are all consenting adults here"** — Python trusts developers to respect the conventions rather than enforcing hard restrictions.

In [ ]:
class Demo:
    def __init__(self):
        self.public_var = "I am public"
        self._protected_var = "I am protected (convention)"
        self.__private_var = "I am private (name-mangled)"

d = Demo()
print(d.public_var)
print(d._protected_var)      # accessible, but you *shouldn't* touch it
# print(d.__private_var)     # AttributeError if uncommented


## 3. Name Mangling

When an attribute name inside a class starts with two underscores (and does **not** end with two underscores), Python automatically renames it internally to `_ClassName__attribute`. This is called **name mangling**.

It doesn't make the attribute truly private — it just makes accidental access or overriding in subclasses much less likely.

In [ ]:
class Demo:
    def __init__(self):
        self.__private_var = 42

d = Demo()
print(d._Demo__private_var)   # works, because of name mangling
try:
    print(d.__private_var)    # fails
except AttributeError as e:
    print("AttributeError:", e)


## 4. Getters and Setters (Traditional Way)

A common pattern in many OOP languages is to keep the attribute private and expose explicit `get_x()` / `set_x()` methods to read and write it, adding validation logic in the setter.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.__name = name
        self.__age = age

    # Getters
    def get_name(self):
        return self.__name

    def get_age(self):
        return self.__age

    # Setters
    def set_name(self, name):
        if not name:
            raise ValueError("Name cannot be empty")
        self.__name = name

    def set_age(self, age):
        if age < 0:
            raise ValueError("Age cannot be negative")
        self.__age = age


p = Person("Alice", 30)
print(p.get_name(), p.get_age())

p.set_age(31)
print(p.get_age())

try:
    p.set_age(-5)
except ValueError as e:
    print("ValueError:", e)


This works, but it is verbose, and it changes the calling code from `person.age` to `person.get_age()` / `person.set_age(x)`, which is less natural in Python. Python offers a cleaner solution: **properties**.

## 5. The `@property` Decorator (The Pythonic Way)

The `@property` decorator lets you define methods that behave like attributes. You can read/write them with normal attribute syntax (`obj.age`), while still running your validation code behind the scenes.

- `@property` — turns a method into a **getter**
- `@<name>.setter` — defines the **setter** for that property
- `@<name>.deleter` — defines what happens on `del obj.<name>` (optional)

In [ ]:
class Person:
    def __init__(self, name, age):
        self.__name = name
        self.__age = age

    @property
    def name(self):
        """Getter for name"""
        return self.__name

    @name.setter
    def name(self, value):
        """Setter for name, with validation"""
        if not value:
            raise ValueError("Name cannot be empty")
        self.__name = value

    @property
    def age(self):
        """Getter for age"""
        return self.__age

    @age.setter
    def age(self, value):
        """Setter for age, with validation"""
        if value < 0:
            raise ValueError("Age cannot be negative")
        self.__age = value


p = Person("Bob", 25)
print(p.name, p.age)      # looks like normal attribute access...

p.age = 26                # ...but this actually calls the setter!
print(p.age)

try:
    p.age = -10            # validation still runs
except ValueError as e:
    print("ValueError:", e)


Notice the calling code (`p.age`, `p.age = 26`) looks exactly like plain attribute access — but validation logic runs automatically. This is the idiomatic way to do encapsulation in Python.

## 6. Computed / Derived Properties

Properties are also useful for values that are **calculated on the fly** rather than stored directly.

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

    @property
    def perimeter(self):
        return 2 * (self.width + self.height)


r = Rectangle(4, 5)
print("Area:", r.area)
print("Perimeter:", r.perimeter)


## 7. Read-Only Properties

If you define only a getter (`@property`) and no setter, the attribute becomes **read-only** from outside the class — attempting to assign to it raises an `AttributeError`.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.__radius = radius

    @property
    def radius(self):
        return self.__radius

    @property
    def area(self):
        return 3.14159 * self.__radius ** 2


c = Circle(3)
print(c.radius, c.area)

try:
    c.area = 100   # no setter defined -> error
except AttributeError as e:
    print("AttributeError:", e)


## 8. Complete Example: `BankAccount`

Let's put everything together in a realistic example. The `BankAccount` class:

- Keeps `__balance` private
- Exposes a read-only `balance` property
- Validates deposits and withdrawals through methods
- Uses a protected `_owner` attribute for internal bookkeeping

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self._owner = owner          # protected: intended for internal/subclass use
        self.__balance = balance     # private: fully controlled by this class

    @property
    def owner(self):
        return self._owner

    @property
    def balance(self):
        """Read-only: balance can only change via deposit()/withdraw()"""
        return self.__balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.__balance += amount
        print(f"Deposited {amount}. New balance: {self.__balance}")

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self.__balance:
            raise ValueError("Insufficient funds")
        self.__balance -= amount
        print(f"Withdrew {amount}. New balance: {self.__balance}")

    def __repr__(self):
        return f"BankAccount(owner={self._owner!r}, balance={self.__balance})"


account = BankAccount("Alice", 100)
print(account)

account.deposit(50)
account.withdraw(30)

try:
    account.withdraw(1000)     # insufficient funds
except ValueError as e:
    print("ValueError:", e)

try:
    account.balance = 999999   # can't bypass the rules directly
except AttributeError as e:
    print("AttributeError:", e)

print(account)


Notice how `account.balance = 999999` fails — the only way to change the balance is through the controlled `deposit()` and `withdraw()` methods, both of which validate the amount. This is encapsulation in action: **the object protects its own invariants.**

## 9. Practice Exercises

Try these on your own in the empty cell(s) below.

1. Create a `Temperature` class that stores temperature internally in **Celsius**, but exposes a `fahrenheit` property (getter and setter) that converts automatically.
2. Create a `Student` class with a private `__grades` list. Add a method `add_grade(score)` that only accepts scores between 0 and 100, and a read-only property `average` that returns the mean grade.
3. Modify the `BankAccount` class above to add a private `__transaction_history` list that records every deposit and withdrawal, exposed via a read-only property `history`.


In [ ]:
# Exercise 1: Temperature (Celsius <-> Fahrenheit)
class Temperature:
    def __init__(self, celsius=0):
        self.__celsius = celsius

    @property
    def celsius(self):
        return self.__celsius

    @celsius.setter
    def celsius(self, value):
        self.__celsius = value

    @property
    def fahrenheit(self):
        return self.__celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.__celsius = (value - 32) * 5 / 9


t = Temperature(25)
print(t.celsius, t.fahrenheit)
t.fahrenheit = 98.6
print(round(t.celsius, 1))


In [ ]:
# Exercise 2: Student grades (write your solution here)




In [ ]:
# Exercise 3: BankAccount with transaction history (write your solution here)




## Summary

- **Encapsulation** bundles data and behavior together and restricts direct, unchecked access to an object's internals.
- Python uses **naming conventions** (`_protected`, `__private`) rather than strict access modifiers.
- `__private` attributes are **name-mangled** to `_ClassName__private`, discouraging (but not fully preventing) outside access.
- The **`@property`** decorator is the idiomatic Python way to add getters/setters while keeping clean attribute-style syntax.
- Properties can also expose **computed values** or be made **read-only** by omitting the setter.
- Good encapsulation lets an object **protect its own invariants** (e.g., a bank balance can never go negative through direct assignment).

**Next up:** `04_Inheritance.ipynb`
